In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


In [ ]:
import os
import sys
from pathlib import Path
import tifffile
import matplotlib.pyplot as plt
import numpy as np

# Add src to path
project_root = Path('..').resolve()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

from spacemap.utils import log_time
from spacemap.background_subtraction_utils import get_channel_info, get_image_mpp
from spacemap.registration import (
    preprocess_he_image, 
    extract_dapi_channel, 
    register_images, 
    register_codex_to_codex,
    merge_multi_round_ome_tiff
)

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")

In [ ]:
# Reload modules to ensure latest changes are picked up
import importlib
from spacemap import registration
from spacemap import utils
from spacemap import background_subtraction_utils

importlib.reload(utils)
importlib.reload(background_subtraction_utils)
importlib.reload(registration)

from spacemap.registration import (
    preprocess_he_image, 
    extract_dapi_channel, 
    register_images, 
    register_codex_to_codex,
    merge_multi_round_ome_tiff
)
print("Modules reloaded successfully.")

In [ ]:
# =============================================================================
# 1. INPUT CONFIGURATION
# =============================================================================
# Define absolute or relative paths to your images
# Round 1 is the ANCHOR (Fixed) image
PATH_CODEX_R1 = f'{DATA_DIR}/example_slides/oph_slide1_codex1.qptiff'

# Round 2 is the MOVING image
PATH_CODEX_R2 = f'{DATA_DIR}/example_slides/oph_slide1_codex2.qptiff'

# H&E Image
PATH_HE = f'{DATA_DIR}/example_slides/oph_slide1_he.qptiff'

# Optional manual anchor overrides. Leave as None to use automatic detection.
# Example: ANCHOR_CHANNEL_R1 = 'Hoechst'
ANCHOR_CHANNEL_R1 = None
ANCHOR_CHANNEL_R2 = None

def detect_mpp(path, fallback):
    if path and os.path.exists(path):
        return float(get_image_mpp(path, default=fallback).get('codex', fallback))
    return float(fallback)

MPP_R1 = detect_mpp(PATH_CODEX_R1, 0.5083)
MPP_R2 = detect_mpp(PATH_CODEX_R2, 0.5083)
MPP_HE = detect_mpp(PATH_HE, 0.2495)

# Output Directory
OUTPUT_DIR = '../outputs/registration_multiround'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Input configuration loaded.")
print(f"Detected MPP R1: {MPP_R1:.4f}")
print(f"Detected MPP R2: {MPP_R2:.4f}")
print(f"Detected MPP H&E: {MPP_HE:.4f}")
print(f"R1 anchor override: {ANCHOR_CHANNEL_R1 or 'Auto'}")
print(f"R2 anchor override: {ANCHOR_CHANNEL_R2 or 'Auto'}")

In [ ]:
# =============================================================================
# 1.1 CHANNEL DISCOVERY
# =============================================================================
# Run this cell to see available channels in your images
# This helps you fill out the CHANNEL CONFIGURATION below

from spacemap.background_subtraction_utils import get_channel_info

def print_image_channels(path, label):
    print(f"\n{'='*40}")
    print(f"Scanning {label}...")
    print(f"Path: {path}")
    
    if not os.path.exists(path):
        print(f" File not found! Please check the path.")
        return

    try:
        # get_channel_info returns 4 values: names, colors, metadata, mpp
        # We use *rest to handle any number of additional return values dynamically
        channel_names, *rest = get_channel_info(path)
        
        print(f"Found {len(channel_names)} channels:")
        for i, name in enumerate(channel_names):
            print(f"  {i}: {name}")
            
        # Print formatted list for easy copy-pasting
        formatted_names = ", ".join([f'"{name}"' for name in channel_names])
        print(f"\n Copy-paste list for configuration:\n{formatted_names}")
        
    except Exception as e:
        print(f" Error reading channels: {e}")
        # import traceback
        # traceback.print_exc()

print_image_channels(PATH_CODEX_R1, "Round 1 (Anchor)")
print_image_channels(PATH_CODEX_R2, "Round 2 (Moving)")
print_image_channels(PATH_HE, "H&E")

In [ ]:
# =============================================================================
# 2. CHANNEL CONFIGURATION
# =============================================================================
# Define which channels to keep and how to rename them
# 'keep': List of channel names to include. None = keep all.
# 'rename': Dictionary mapping old names to new names.
# 'colors': Dictionary mapping original names to RGB tuples (0-255).

CHANNEL_CONFIG = [
    # Config for Round 1 (Index 0)
    {
        "keep": ["DAPI", "CD44", "CD31", "CD107a", "E-cadherin", "CD4", "CD8", "CD45", "CD68", "Ki67", "CD11c", "aSMA", "HLA-DR", "FN1", "GP100", "gH2AX", "FoxP3", "PD-L1", "CD163", "CD40", "CD11b", "Blank15 Atto550", "Blank15 AF750", "Blank15 AF647"], 
        "rename": {
            "CD8": "CD8_R1"
        }
    },
    # Config for Round 2 (Index 1)
    {
        "keep": [ "CD8", "CD20", "CD3e"], 
        "rename": {
             # Rename to avoid collision with R1 DAPI
        }
    },
    # Config for H&E (Index 2)
    {
        # For H&E, if the input is a QPTIFF or standard RGB, it may not have named channels.
        # In that case, set "keep": None to include all 3 channels (Red, Green, Blue).
        # You can use "rename" to map generic names (e.g., "Channel_0") to "HE_Red", etc.
        # If the input is a pre-registered OME-TIFF with named channels, you can list them in "keep".
        "keep": None, 
        "rename": {
            "Channel_0": "HE_Red",
            "Channel_1": "HE_Green",
            "Channel_2": "HE_Blue"
        },
        "colors": {
            "Channel_0": (255, 0, 0),   # Red
            "Channel_1": (0, 255, 0),   # Green
            "Channel_2": (0, 0, 255)    # Blue
        }
    }
]

print("Channel configuration loaded.")

In [ ]:
# =============================================================================
# 3. PREPROCESSING & REFERENCE EXTRACTION
# =============================================================================
log_time("Starting preprocessing...")

# Extract registration references. For pyramidal OME-TIFF inputs this may reuse the
# source image directly instead of materializing a separate single-channel TIFF.
r1_dapi_path = extract_dapi_channel(PATH_CODEX_R1, OUTPUT_DIR, reference_channel=ANCHOR_CHANNEL_R1)
r2_dapi_path = extract_dapi_channel(PATH_CODEX_R2, OUTPUT_DIR, reference_channel=ANCHOR_CHANNEL_R2)

# Resize H&E when needed. If the H&E image is already a pyramidal OME-TIFF at
# the target resolution, preprocess_he_image returns the original path.
mpp_ratio = {
    'hne': MPP_HE,
    'codex': MPP_R1
}
he_resized_path = preprocess_he_image(PATH_HE, OUTPUT_DIR, mpp_ratio)

print(f"R1 reference: {r1_dapi_path}")
print(f"R2 reference: {r2_dapi_path}")
print(f"H&E input for registration: {he_resized_path}")

In [ ]:
# =============================================================================
# 4. REGISTRATION
# =============================================================================
log_time("Starting registration...")

def normalized_base_name(path, suffix):
    name = Path(path).name
    for ext in ('.ome.tiff', '.ome.tif', '.qptiff', '.tiff', '.tif'):
        if name.lower().endswith(ext):
            name = name[:-len(ext)]
            break
    if suffix and name.endswith(suffix):
        name = name[:-len(suffix)]
    return name

# A. Register R2 to R1
base_r1 = normalized_base_name(PATH_CODEX_R1, '_codex')
base_r2 = normalized_base_name(PATH_CODEX_R2, '_codex')
base_he = normalized_base_name(PATH_HE, '_he')
r2_registered_path = os.path.join(OUTPUT_DIR, f"{base_r2}_registered_to_R1.ome.tiff")
mpp_reg = {'codex_ref': MPP_R1, 'codex_mov': MPP_R2}

register_codex_to_codex(
    fixed_dapi_path=r1_dapi_path,
    moving_dapi_path=r2_dapi_path,
    moving_codex_path=PATH_CODEX_R2,
    output_path=r2_registered_path,
    mpp=mpp_reg,
    fixed_reference_channel=ANCHOR_CHANNEL_R1,
    moving_reference_channel=ANCHOR_CHANNEL_R2,
 )

# B. Register H&E to R1
he_registered_path = os.path.join(OUTPUT_DIR, f"{base_he}_registered_to_R1.ome.tiff")
mpp_he_reg = {'codex': MPP_R1, 'hne': MPP_R1}  # Both are now at R1 scale

register_images(
    he_resized_path=he_resized_path,
    dapi_path=r1_dapi_path,
    output_dir=OUTPUT_DIR,
    mpp=mpp_he_reg,
    reference_channel=ANCHOR_CHANNEL_R1,
 )
# Note: register_images returns the path, but we can construct it
he_registered_actual = os.path.join(OUTPUT_DIR, f"{base_he}_he_registered.ome.tiff")

print("Registration completed.")

In [ ]:
# =============================================================================
# 5. MERGING
# =============================================================================
log_time("Starting merging...")

final_output_path = os.path.join(OUTPUT_DIR, f"{base_r1}_multiround_merged.ome.tiff")

# List of images to merge
# 1. R1 (Original)
# 2. R2 (Registered)
# 3. H&E (Registered)
images_to_merge = [
    PATH_CODEX_R1,
    r2_registered_path,
    he_registered_actual
]

merge_multi_round_ome_tiff(
    image_paths=images_to_merge,
    output_path=final_output_path,
    mpp=MPP_R1,
    channel_configs=CHANNEL_CONFIG
)

print(f"Done! Final output: {final_output_path}")